In [46]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils import resample  
import warnings
import os
warnings.filterwarnings('ignore')


In [47]:
print("Loading dataset...")
print("Current directory:", os.getcwd())
print("CSV files found:", [f for f in os.listdir('.') if f.endswith('.csv')])

# Load with robust parsing
try:
    df = pd.read_csv('benign_vs_defacement_urls.csv', 
                     on_bad_lines='skip',
                     quoting=3,
                     engine='python')
except:
    # Manual parsing fallback
    data = []
    with open('benign_vs_defacement_urls.csv', 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f, 1):
            parts = line.strip().split(',', 1)
            if len(parts) == 2:
                data.append(parts)
    
    df = pd.DataFrame(data, columns=['url', 'label'])

print("Dataset shape:", df.shape)
print("\nColumn names found:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))
print("\nData types:")
print(df.dtypes)

print("\nUnique values in column 1:", df.iloc[:, 1].unique()[:10])

# Handle different possible column structures
if len(df.columns) >= 2:
    # Rename columns safely
    df.columns = ['url', 'label']
    
    # Clean data
    df = df.dropna()
    df = df[df['url'].str.len() > 0]
    
    # Map labels - handle variations safely
    unique_labels = df['label'].unique()
    print("Unique labels before mapping:", unique_labels)
    
    label_mapping = {'benign': 'good', 'defacement': 'bad'}
    df['label'] = df['label'].str.strip().str.lower().map(label_mapping)
    
    # Fill any unmapped labels with original (shouldn't happen)
    df['label'] = df['label'].fillna(df['label'])
    
    print("\nClean dataset shape:", df.shape)
    print("Final label distribution:")
    print(df['label'].value_counts())
    
else:
    print("ERROR: Dataset doesn't have expected 2 columns")
    print("Available columns:", df.columns.tolist())


Loading dataset...
Current directory: c:\Users\sunny\Academics\Sem-6\ML\Lab\Lab2
CSV files found: ['benign_vs_defacement_urls.csv']
Dataset shape: (524003, 2)

Column names found: ['url', 'type']

First 3 rows:
                                                 url    type
0                mp3raid.com/music/krizz_kaliko.html  benign
1                    bopsecrets.org/rexroth/cr/1.htm  benign
2  http://buzzfil.net/m/show-art/ils-etaient-loin...  benign

Data types:
url     object
type    object
dtype: object

Unique values in column 1: ['benign' 'defacement' None]
Unique labels before mapping: ['benign' 'defacement']

Clean dataset shape: (523934, 2)
Final label distribution:
label
good    427883
bad      96051
Name: count, dtype: int64


In [48]:
def extract_url_features(url):
    """Extract all features from a URL"""
    features = {}
    
    # 1. Length of full URL
    features['url_length'] = len(url)
    
    # 2. Hostname length
    try:
        parsed = urlparse(url)
        hostname = parsed.netloc
        features['hostname_length'] = len(hostname)
    except:
        features['hostname_length'] = 0
    
    # 3. Symbol counting
    features['count_dot'] = url.count('.')
    features['count_dash'] = url.count('-')
    features['count_at'] = url.count('@')
    features['count_question'] = url.count('?')
    features['count_percent'] = url.count('%')
    features['count_equal'] = url.count('=')
    
    # 4. Digit count
    features['digit_count'] = sum(c.isdigit() for c in url)
    
    # 5. Special patterns
    # Check if URL uses IP address
    ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
    features['has_ip'] = 1 if re.search(ip_pattern, url) else 0
    
    # Check for shortened URL services
    shorteners = ['bit.ly', 'goo.gl', 'tinyurl.com', 't.co', 'ow.ly', 'is.gd']
    features['is_shortened'] = 1 if any(short in url.lower() for short in shorteners) else 0
    
    # Count slashes (directories)
    features['slash_count'] = url.count('/')
    
    return features

# Test the function
test_url = "https://www.example.com/page?id=123"
print("Testing feature extraction:")
print(extract_url_features(test_url))


Testing feature extraction:
{'url_length': 35, 'hostname_length': 15, 'count_dot': 2, 'count_dash': 0, 'count_at': 0, 'count_question': 1, 'count_percent': 0, 'count_equal': 1, 'digit_count': 3, 'has_ip': 0, 'is_shortened': 0, 'slash_count': 3}


In [49]:
# Extract features for all URLs
print("Extracting features from all URLs...")
feature_list = []

for url in df['url']:
    features = extract_url_features(url)
    feature_list.append(features)

# Create feature dataframe
features_df = pd.DataFrame(feature_list)

# Combine with labels
df_with_features = pd.concat([df[['url', 'label']], features_df], axis=1)

print("\nDataset with extracted features:")
print(df_with_features.head())
print("\nFeature columns:", features_df.columns.tolist())
print("\nDataset shape:", df_with_features.shape)


Extracting features from all URLs...

Dataset with extracted features:
                                                 url label  url_length  \
0                mp3raid.com/music/krizz_kaliko.html  good        35.0   
1                    bopsecrets.org/rexroth/cr/1.htm  good        31.0   
2  http://buzzfil.net/m/show-art/ils-etaient-loin...  good       118.0   
3      espn.go.com/nba/player/_/id/3457/brandon-rush  good        45.0   
4     yourbittorrent.com/?q=anthony-hamilton-soulife  good        46.0   

   hostname_length  count_dot  count_dash  count_at  count_question  \
0              0.0        2.0         0.0       0.0             0.0   
1              0.0        2.0         0.0       0.0             0.0   
2             11.0        2.0        16.0       0.0             0.0   
3              0.0        2.0         1.0       0.0             0.0   
4              0.0        1.0         2.0       0.0             1.0   

   count_percent  count_equal  digit_count  has_ip  is_sh

In [50]:
# CRITICAL FIX: Align X and y by using df_with_features directly
print("Original dataset shapes:")
print("df_with_features shape:", df_with_features.shape)
print("features_df shape:", features_df.shape)

# Use ONLY the overlapping indices - truncate to minimum length
min_rows = min(features_df.shape[0], df_with_features.shape[0])
X = features_df.iloc[:min_rows].reset_index(drop=True)
y = df_with_features['label'].iloc[:min_rows].reset_index(drop=True)

print(f"\nAligned shapes - X: {X.shape}, y: {y.shape}")
print("Original class distribution:")
print(y.value_counts())

# Convert all features to float (already numeric but ensure consistency)
X = X.astype(float)

# Check for NaN values and remove
nan_mask = X.isna().any(axis=1)
if nan_mask.sum() > 0:
    print(f"Removing {nan_mask.sum()} rows with NaN features")
    X = X[~nan_mask].reset_index(drop=True)
    y = y[~nan_mask].reset_index(drop=True)

print(f"Clean aligned shapes - X: {X.shape}, y: {y.shape}")
print("Clean class distribution:")
print(y.value_counts())

# Manual oversampling
from sklearn.utils import resample
import numpy as np

# Separate classes
good_mask = (y == 'good')
bad_mask = (y == 'bad')

X_good = X[good_mask]
X_bad = X[bad_mask]
y_good = y[good_mask]
y_bad = y[bad_mask]

print(f"\nMajority class (good): {len(X_good)} samples")
print(f"Minority class (bad): {len(X_bad)} samples")

# Oversample bad class to match good class
X_bad_upsampled = resample(X_bad, 
                          replace=True, 
                          n_samples=len(X_good), 
                          random_state=42)
y_bad_upsampled = resample(y_bad, 
                          replace=True, 
                          n_samples=len(y_good), 
                          random_state=42)

# Combine
X_resampled = np.vstack([X_good.values, X_bad_upsampled.values])
y_resampled = np.hstack([y_good.values, y_bad_upsampled.values])

# Shuffle
shuffle_idx = np.random.permutation(len(X_resampled))
X_resampled = X_resampled[shuffle_idx]
y_resampled = y_resampled[shuffle_idx]

print("\nResampled class distribution:")
unique, counts = np.unique(y_resampled, return_counts=True)
for label, count in zip(unique, counts):
    print(f"{label}: {count}")
print("Final resampled shape:", X_resampled.shape)


Original dataset shapes:
df_with_features shape: (524003, 14)
features_df shape: (523934, 12)

Aligned shapes - X: (523934, 12), y: (523934,)
Original class distribution:
label
good    427883
bad      96051
Name: count, dtype: int64
Clean aligned shapes - X: (523934, 12), y: (523934,)
Clean class distribution:
label
good    427883
bad      96051
Name: count, dtype: int64

Majority class (good): 427883 samples
Minority class (bad): 96051 samples

Resampled class distribution:
bad: 427883
good: 427883
Final resampled shape: (855766, 12)


In [51]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, 
    test_size=0.2, 
    random_state=42,
    stratify=y_resampled
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

# Train Logistic Regression model
print("\nTraining Logistic Regression model...")
model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model training completed!")


Training set size: (684612, 12)
Test set size: (171154, 12)

Training Logistic Regression model...
Model training completed!


In [52]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("MODEL EVALUATION RESULTS")
print(f"\nAccuracy Score: {accuracy:.2%}")
print(f"The model correctly classified {accuracy:.2%} of URLs")

# Confusion Matrix
print("CONFUSION MATRIX")
cm = confusion_matrix(y_test, y_pred, labels=['good', 'bad'])
print("\n", cm)

# Interpret confusion matrix
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (Good sites correctly identified): {tn}")
print(f"False Positives (Good sites incorrectly blocked): {fp}")
print(f"False Negatives (Bad sites that were missed): {fn}")
print(f"True Positives (Bad sites correctly caught): {tp}")

# Classification Report
print("DETAILED CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))


MODEL EVALUATION RESULTS

Accuracy Score: 97.21%
The model correctly classified 97.21% of URLs
CONFUSION MATRIX

 [[82722  2855]
 [ 1914 83663]]

Confusion Matrix Breakdown:
True Negatives (Good sites correctly identified): 82722
False Positives (Good sites incorrectly blocked): 2855
False Negatives (Bad sites that were missed): 1914
True Positives (Bad sites correctly caught): 83663
DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

         bad       0.97      0.98      0.97     85577
        good       0.98      0.97      0.97     85577

    accuracy                           0.97    171154
   macro avg       0.97      0.97      0.97    171154
weighted avg       0.97      0.97      0.97    171154

